# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

In [2]:
# importar librerías
import pandas as pd
import numpy as nm
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
# cargar archivos
orders = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv")
catalog = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv")
marketing = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv")

In [4]:
# explorar datasets

orders.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [5]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [6]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [7]:


orders.head(10)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28
5,order_5,user_2792,2025-04-22,mexico,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,179.34,0.0,179.34
6,order_6,user_2802,2025-03-15,Colombia,mobile,organic,Blender-XL-Red,Hogar,2.0,163.59,0.0,327.19
7,order_7,user_1083,2025-05-01,Mexico,desktop,organic,Tablet-Standard-64GB,Electronica,2.0,373.68,0.0,747.35
8,order_8,user_2352,2025-03-01,Argentina,desktop,paid_search,Laptop-Gaming-16GB,Electronica,2.0,477.27,5.0,949.54
9,order_9,user_7001,2025-05-31,Mexico,mobile,organic,Sneakers-Urban-42,Moda,1.0,339.30,5.0,334.30


In [8]:

marketing["fecha"]=pd.to_datetime(marketing["fecha"],errors="coerce")
orders["fecha_hora_pedido"]=pd.to_datetime(orders["fecha_hora_pedido"],errors="coerce")
orders = orders[orders['cantidad'] < 100]




In [9]:
orders[["precio_unitario","cantidad","monto_descuento","monto_total"]].describe()

,precio_unitario,cantidad,monto_descuento,monto_total
count,25040.000000,25040.000000,25040.000000,25040.000000
mean,259.295062,1.504513,4.502596,385.700062
std,138.728216,0.501186,5.223278,255.881692
min,20.030000,-2.000000,0.000000,-492.650000
25%,138.367500,1.000000,0.000000,180.285000
50%,258.700000,2.000000,0.000000,341.315000
75%,380.355000,2.000000,10.000000,517.630000
max,499.960000,2.000000,15.000000,999.890000


In [10]:
cantidad_mediana=orders["cantidad"].median()
orders["cantidad"]=orders["cantidad"].replace(-2,cantidad_mediana)
orders["cantidad"]=orders["cantidad"].replace(-1,cantidad_mediana)
monto_mediana=orders["monto_total"].median()
orders.loc[orders["monto_total"] <= 0, "monto_total"] = monto_mediana


In [11]:
marketing.head(5)

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [12]:
marketing["gasto"].describe()

count    1620.00000
mean     1772.74292
std       734.43294
min       501.11000
25%      1128.03000
50%      1782.42500
75%      2420.68500
max      2999.36000
Name: gasto, dtype: float64

In [13]:
catalog.head(5)

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [14]:
catalog["costo_unitario"].describe()

count      7.000000
mean     102.252857
std      111.011563
min       10.120000
25%       16.905000
50%       25.210000
75%      182.975000
max      280.680000
Name: costo_unitario, dtype: float64

In [15]:
orders['monto_total'] = (orders['cantidad'] * orders['precio_unitario']) - orders['monto_descuento']

In [16]:
orders=orders.drop_duplicates(subset="id_pedido")
print(orders["id_pedido"].duplicated().sum())

0


In [17]:
orders[["pais","dispositivo","fuente_referencia","nombre_producto","categoria_producto"]].value_counts()

pais       dispositivo  fuente_referencia  nombre_producto       categoria_producto
Argentina  mobile       organic            Sneakers-Urban-42     Moda                  244
Colombia   desktop      social             Vacuum-Pro-Black      Hogar                 243
                        organic            Jacket-Winter-M       Moda                  238
                        social             Jacket-Winter-M       Moda                  237
                        paid_search        Blender-XL-Red        Hogar                 230
                                                                                      ... 
mexico     desktop      social             Tablet-Standard-64GB  Electronica            11
colombia   desktop      social             Tablet-Standard-64GB  Electronica            10
mexico     mobile       paid_search        Tablet-Standard-64GB  Electronica             9
colombia   mobile       social             Tablet-Standard-64GB  Electronica             8
argent

In [18]:
orders["pais"].isna().sum()

300

In [19]:
columnas_o_categoricas=["pais","dispositivo","fuente_referencia","nombre_producto","categoria_producto"]
for columnas in columnas_o_categoricas:
    orders=orders.dropna(subset=[columnas])
    print(orders[columnas].isna().sum())


0
0
0
0
0


In [20]:
columnas_o_categoricas=["pais","dispositivo","fuente_referencia","nombre_producto","categoria_producto"]
print(orders[columnas_o_categoricas].describe())

            pais dispositivo fuente_referencia  nombre_producto  \
count      24590       24590             24590            24590   
unique         6           2                 3                7   
top     Colombia     desktop            social  Jacket-Winter-M   
freq        7450       12519              8265             4122   

       categoria_producto  
count               24590  
unique                  3  
top                 Hogar  
freq                 8228  


In [21]:

columnas_c_categoricas=["nombre_producto","categoria_producto","proveedor"]
for columnas in columnas_c_categoricas:
    catalog=catalog.dropna(subset=[columnas])
    print(catalog[columnas].isna().sum())
print(catalog[columnas_c_categoricas].describe())

0
0
0
        nombre_producto categoria_producto   proveedor
count                 7                  7           7
unique                7                  3           7
top     Phone-Pro-128GB        Electrónica  Bowers LLC
freq                  1                  3           1


In [22]:
columnas_m_categoricas=["pais","id_campaña","canal"]
for columnas in columnas_m_categoricas:
    marketing=marketing.dropna(subset=[columnas])
    print(marketing[columnas].isna().sum())
print(marketing[columnas_m_categoricas].describe())



0
0
0
            pais        id_campaña        canal
count       1519              1519         1519
unique         3                 9            3
top     Colombia  organic_Colombia  paid_search
freq         507               169          507


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [23]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [24]:



revenue=sum(orders["monto_total"])
print(revenue)


9491675.920000028


In [25]:

orders_con_costo = orders.merge(
    catalog[['nombre_producto', 'costo_unitario']], 
    on='nombre_producto', 
    how='left'
)
revenue=orders["monto_total"].sum()
# Calcular el costo total de cada pedido (cantidad * costo_unitario)
orders_con_costo['costo_orden'] = orders_con_costo['cantidad'] * orders_con_costo['costo_unitario']
gasto_marketing = marketing['gasto'].sum()
# Sumar el costo de todos los pedidos (Costo de ventas / COGS)
costo_productos = orders_con_costo['costo_orden'].sum()
costos_totales=costo_productos+gasto_marketing
print(f"Costo total de productos vendidos: {costo_productos:,.2f}")
print(f"Inversión total en marketing: {gasto_marketing:,.2f}")
profit=revenue-costos_totales
print(f"EL profit es: {profit}")
ticket_p = revenue / orders["id_pedido"].nunique()
print(f"El ticket promedio es: ${ticket_p:,.2f}")
cantidad_p=orders["cantidad"].sum()/orders["id_pedido"].count()
print(f"La cantidad promedio por orden es: {cantidad_p}")
print("El producto más vendido es:", orders.groupby("nombre_producto")["id_pedido"].count().idxmax())
marketing.groupby("canal")["gasto"].sum()


    


Costo total de productos vendidos: 3,783,478.80
Inversión total en marketing: 2,694,664.43
EL profit es: 3013532.6900000004
El ticket promedio es: $386.00
La cantidad promedio por orden es: 1.5052053680357869
EL producto más vendido es
El producto más vendido es: Jacket-Winter-M


canal
organic        913533.01
paid_search    863088.21
social         918043.21
Name: gasto, dtype: float64

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [26]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Configuración de conexión a PostgreSQL
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': '••••••••',
    'host': 'db-host.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [27]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head(5)


,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [28]:
query_names="""
SELECT nombre_evento, COUNT(*) AS total
FROM events
GROUP BY nombre_evento;
"""

nombres=pd.read_sql(query_names, con=engine)
print(nombres)

      nombre_evento  total
0  add_payment_info  12018
1       first_visit  29957
2    begin_checkout  17971
3       add_to_cart  24157
4       select_item  23887
5          purchase  12010


In [29]:
# PARTE 1: Totales del funnel
# ======================

query_totals = """
WITH first_visits AS (
  SELECT DISTINCT id_usuario		
  FROM events
  WHERE nombre_evento = 'first_visit'
),
select_item AS (
  SELECT DISTINCT id_usuario		
  FROM events
  WHERE nombre_evento = 'select_item'
),
add_to_cart AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_to_cart'
),
begin_checkout AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'begin_checkout'
),
 add_payment_info AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_payment_info'
),
purchase AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'purchase'
)
SELECT
  COUNT(fv.id_usuario) AS usuarios_first_visit,
  COUNT(si.id_usuario) AS usuarios_select_item,
  COUNT(a.id_usuario) AS usuarios_add_to_cart,
  COUNT(bc.id_usuario) AS usuarios_begin_checkout,
  COUNT(api.id_usuario) AS usuarios_add_payment_info,
  COUNT(p.id_usuario) AS usuarios_purchase
FROM first_visits fv
LEFT JOIN select_item si        ON fv.id_usuario = si.id_usuario 
LEFT JOIN add_to_cart a         ON fv.id_usuario = a.id_usuario
LEFT JOIN begin_checkout bc     ON fv.id_usuario= bc.id_usuario 
LEFT JOIN add_payment_info api  ON fv.id_usuario = api.id_usuario
LEFT JOIN purchase p            ON fv.id_usuario = p.id_usuario
"""
totals = pd.read_sql(query_totals, con=engine)
totals

,usuarios_first_visit,usuarios_select_item,usuarios_add_to_cart,usuarios_begin_checkout,usuarios_add_payment_info,usuarios_purchase
0,7796,7393,7439,7031,6088,6068


In [30]:
# PARTE 2: Conversiones
# ======================

query_conversion ="""
WITH first_visits AS (
  SELECT DISTINCT id_usuario		
  FROM events
  WHERE nombre_evento = 'first_visit'
),
select_item AS (
  SELECT DISTINCT id_usuario		
  FROM events
  WHERE nombre_evento = 'select_item'
),
add_to_cart AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_to_cart'
),
begin_checkout AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'begin_checkout'
),
 add_payment_info AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_payment_info'
),
purchase AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'purchase'
),
funnel_counts AS(
SELECT
  COUNT(fv.id_usuario) AS usuarios_first_visit,
  COUNT(si.id_usuario) AS usuarios_select_item,
  COUNT(a.id_usuario) AS usuarios_add_to_cart,
  COUNT(bc.id_usuario) AS usuarios_begin_checkout,
  COUNT(api.id_usuario) AS usuarios_add_payment_info,
  COUNT(p.id_usuario) AS usuarios_purchase
FROM first_visits fv
LEFT JOIN select_item si        ON fv.id_usuario = si.id_usuario 
LEFT JOIN add_to_cart a         ON fv.id_usuario = a.id_usuario
LEFT JOIN begin_checkout bc     ON fv.id_usuario= bc.id_usuario 
LEFT JOIN add_payment_info api  ON fv.id_usuario = api.id_usuario
LEFT JOIN purchase p            ON fv.id_usuario = p.id_usuario
)
SELECT 
    usuarios_select_item*100.0/NULLIF(usuarios_first_visit,0) AS conversion_select_item,
 usuarios_add_to_cart*100.0/NULLIF(usuarios_first_visit,0) AS conversion_add_to_cart,
 usuarios_begin_checkout*100.0/NULLIF(usuarios_first_visit,0) AS conversion_begin_checkout,
 usuarios_add_payment_info*100.0/NULLIF(usuarios_first_visit,0) AS conversion_add_payment_info,
 usuarios_purchase*100.0/NULLIF(usuarios_first_visit,0) AS conversion_purchase
FROM 
funnel_counts
    ORDER BY 
    conversion_purchase DESC
"""

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,conversion_select_item,conversion_add_to_cart,conversion_begin_checkout,conversion_add_payment_info,conversion_purchase
0,94.830682,95.420729,90.187276,78.091329,77.834787


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [31]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [32]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT*
FROM user_activity
;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [33]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohort AS (
SELECT
id_usuario,
TO_CHAR(DATE_TRUNC('month', MIN(CAST(fecha_registro AS DATE))), 'YYYY-MM') AS cohort
FROM users
GROUP BY id_usuario
),
activity AS (
    SELECT
    ua.id_usuario, 
    c.cohort, 
    ua.dias_despues_registro,
    ua.activo
    FROM 
    users AS u
    LEFT JOIN 
    cohort AS c ON  u.id_usuario=c.id_usuario
    LEFT JOIN user_activity ua ON u.id_usuario=ua.id_usuario
)
 SELECT 
    cohort,
    ROUND(100.0*COUNT(DISTINCT CASE WHEN dias_despues_registro >= 7 AND activo = 1 THEN id_usuario END)/NULLIF(COUNT(DISTINCT id_usuario),0),1) AS retention_d7_pct,
    ROUND(100.0*COUNT(DISTINCT CASE WHEN dias_despues_registro >= 14 AND activo = 1 THEN id_usuario END)/NULLIF(COUNT(DISTINCT id_usuario),0),1) AS retention_d14_pct,
    ROUND(100.0*COUNT(DISTINCT CASE WHEN dias_despues_registro >= 21 AND activo = 1 THEN id_usuario END)/NULLIF(COUNT(DISTINCT id_usuario),0),1) AS retention_d21_pct,
    ROUND(100.0*COUNT(DISTINCT CASE WHEN dias_despues_registro >= 28 AND activo = 1 THEN id_usuario END)/NULLIF(COUNT(DISTINCT id_usuario),0),1) AS retention_d28_pct 
FROM activity
GROUP BY cohort
ORDER BY cohort;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohort,retention_d7_pct,retention_d14_pct,retention_d21_pct,retention_d28_pct
0,2025-01,84.9,77.0,63.1,41.2
1,2025-02,86.9,79.9,65.1,39.8
2,2025-03,87.3,79.8,64.8,41.1
3,2025-04,86.8,78.5,63.6,40.6
4,2025-05,85.7,78.3,64.5,40.2


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** No existe una diferencia estadísticamente significativa en la tasa de conversión de compra entre la versión actual del checkout (control) y la nueva interfaz (tratamiento).
   - **H₁ (Hipótesis alternativa):** Existe una diferencia estadísticamente significativa en la tasa de conversión de compra entre el grupo control y el grupo tratamiento
   
**Test estadístico:**  Prueba Chi-cuadrado de independencia (al ser una variable binaria/categórica).  
**Nivel de significancia alpha:** 0.05 (5%). Si el valor p es menor a alpha, rechazamos la hipótesis nula (H_0).

In [34]:

experimento = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')



In [35]:
experimento.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [36]:
from scipy.stats import chi2_contingency
alpha=0.05
tabla = pd.crosstab(experimento['variante'], experimento['convirtio'])
tabla_normalizada =pd.crosstab(experimento['variante'], experimento['convirtio'],normalize="index")*100
chi2_stat, p_value, dof, expected = chi2_contingency(tabla)
print(f"Estadístico chi-cuadrado: {chi2_stat}")
print(f"Valor P: {p_value}")
alpha = 0.05  # umbral de significancia
if p_value < alpha:
    print("Rechazamos la hipótesis nula")
else:
    print("No rechazamos la hipótesis nula")

Estadístico chi-cuadrado: 0.6177857659309294
Valor P: 0.43187130118526496
No rechazamos la hipótesis nula


**Conclusión estadística:** No hay evidencia estadísticamente sólida que asegure que la nueva página tenga una tasa de conversión mayor a la original.

* **Recomendación de negocio:** Se recomienda **no implementar (no hacer rollout)** la nueva versión de la UI del checkout para el total de los usuarios, ya que no genera el impacto esperado en ventas y representaría un costo de despliegue innecesario. Dado que el análisis del funnel (Paso 3) reveló que el mayor abandono ocurre al ingresar la información de pago (`begin_checkout` $\rightarrow$ `add_payment_info`), se sugiere replantear la hipótesis de diseño e iterar con un nuevo experimento enfocado en simplificar los métodos de pago o reducir la fricción en ese formulario específico.
    

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 📊 Dashboards Interactivos (Tableau)

Visualización interactiva de KPIs principales, rentabilidad y análisis detallado de ventas:

### 📊 Dashboards del Proyecto
- [Dashboard 1: Overview Ejecutivo en Tableau Public](https://public.tableau.com/app/profile/jorge.moreno6260/viz/overview-rappi/overview?publish=yes)
- [Dashboard 2: Detalle y Drill-down en Tableau Public](https://public.tableau.com/app/profile/jorge.moreno6260/viz/detalle-rappi/detalle?publish=yes)